# TP PAC-Bayes -- Session 2: Learning the weights: optimized certificates, direct minimization, diversity, model selection

Solution notebook for Session 2 (three hours, Parts 2.1 to 2.4 of the sheet `tp_pacbayes.pdf`). It reproduces the numbers, tables (written to `results/`) and figures (written to `figures/`) of the solution sheet `tp_pacbayes_solutions.pdf`.

It needs `pbtools.py`, `prepdata.py` and the folder `datasets/` of the course, either in this folder or in its parent folder. Each session notebook is self-contained: the helper functions it needs are redefined at the top.

In [1]:
import os, sys, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
for p in [HERE, os.path.join(HERE, ".."), os.path.join(HERE, "..", "data"), os.path.join(HERE, "..", "..", "common"), os.path.join(HERE, "data"), os.path.join(HERE, "..", "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)
from pbtools import (kl_bin, kl_inv_upper, kl_inv_lower, kl_div, bound_seeger, bound_mcallester,
                     BaggedTrees, mv_statistics, fo_bound, tnd_bound, cbound_pb, softmax,
                     optimize_fo, optimize_tnd)
from prepdata import data_recovery

DATADIR = next((d for d in [os.path.join(HERE, "data"), HERE, os.path.join(HERE, ".."), os.path.join(HERE, "..", "data")] if os.path.isdir(os.path.join(d, "datasets"))), HERE)
RES = os.path.join(HERE, "results"); FIG = os.path.join(HERE, "figures")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True)
DELTA = 0.05
N_SPLITS = 5
BALANCED = ["australian", "balance", "bupa", "german", "heart", "iono", "pima", "sonar", "wdbc", "spambase"]
IMBALANCED = ["abalone8", "pageblocks", "satimage", "segmentation", "yeast3"]
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})


def load(name):
    """Load a dataset of the course with labels in {-1,+1} and standardized features."""
    cwd = os.getcwd(); os.chdir(DATADIR)
    try:
        X, y = data_recovery(name)
    finally:
        os.chdir(cwd)
    X = StandardScaler().fit_transform(np.asarray(X, dtype=float))
    return X, np.where(np.asarray(y) > 0, 1, -1)


def eff(rho):
    """Effective number of voters 1/sum rho^2."""
    return 1.0 / np.sum(rho ** 2)


def save_tab(name, rows):
    open(os.path.join(RES, name), "w").write("\n".join(rows) + "\n")
    print("  -> results/" + name)


def ms(a, d=3):
    a = np.asarray(a, dtype=float)
    return f"{a.mean():.{d}f} ({a.std():.{d}f})"

## Helpers from Session 1 (random forests with out-of-bag statistics)

In [2]:
def forest_stats(X, y, seed, n_trees=100, frac=0.5, max_features="sqrt", max_depth=None):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
    bt = BaggedTrees(n_trees, bootstrap_frac=frac, max_features=max_features, max_depth=max_depth,
                     random_state=seed).fit(Xtr, ytr)
    L, n1 = bt.oob_losses(Xtr, ytr); Lt, n2 = bt.oob_tandem(Xtr, ytr); D, n3 = bt.oob_disagreement(Xtr)
    return dict(bt=bt, Xtr=Xtr, Xte=Xte, ytr=ytr, yte=yte, L=L, n1=n1, Lt=Lt, n2=n2, D=D, n3=n3,
                Vte=bt.predict_all(Xte), pi=np.ones(n_trees) / n_trees)


def certificates(F, rho):
    return (fo_bound(rho, F["pi"], F["L"], F["n1"], DELTA), tnd_bound(rho, F["pi"], F["Lt"], F["n2"], DELTA),
            cbound_pb(rho, F["pi"], F["L"], F["n1"], F["D"], F["n3"], DELTA))


CACHE = {}


def part2_3():
    print("Part 2 and 3")
    rows2, rows3, summary = [], [], {k: [] for k in ["u", "fo", "tnd"]}
    for name in BALANCED + IMBALANCED:
        X, y = load(name)
        R = {k: [] for k in ["mv", "gibbs", "dis", "fo", "tnd", "cb", "rf", "n1", "n2",
                             "mv_fo", "cert_fo", "eff_fo", "mv_tnd", "cert_tnd", "eff_tnd", "best_u"]}
        for seed in range(N_SPLITS):
            F = forest_stats(X, y, seed)
            CACHE[(name, seed)] = F
            st = mv_statistics(F["Vte"], F["yte"], F["pi"])
            fo, tnd, cb = certificates(F, F["pi"])
            rf = RandomForestClassifier(100, random_state=seed).fit(F["Xtr"], F["ytr"])
            for k, v in [("mv", st["mv"]), ("gibbs", st["gibbs"]), ("dis", st["dis"]), ("fo", fo), ("tnd", tnd),
                         ("cb", cb), ("rf", np.mean(rf.predict(F["Xte"]) != F["yte"])), ("n1", F["n1"]), ("n2", F["n2"])]:
                R[k].append(v)
            # Part 3: learning the weights
            r_fo = optimize_fo(F["pi"], F["L"], F["n1"], DELTA)
            r_tnd = optimize_tnd(F["pi"], F["Lt"], F["n2"], DELTA)
            R["mv_fo"].append(mv_statistics(F["Vte"], F["yte"], r_fo)["mv"])
            R["cert_fo"].append(fo_bound(r_fo, F["pi"], F["L"], F["n1"], DELTA)); R["eff_fo"].append(eff(r_fo))
            R["mv_tnd"].append(mv_statistics(F["Vte"], F["yte"], r_tnd)["mv"])
            R["cert_tnd"].append(tnd_bound(r_tnd, F["pi"], F["Lt"], F["n2"], DELTA)); R["eff_tnd"].append(eff(r_tnd))
            # best of the three certificates of the uniform vote, with a union bound (delta/3)
            R["best_u"].append(min(fo_bound(F["pi"], F["pi"], F["L"], F["n1"], DELTA / 3),
                                   tnd_bound(F["pi"], F["pi"], F["Lt"], F["n2"], DELTA / 3),
                                   cbound_pb(F["pi"], F["pi"], F["L"], F["n1"], F["D"], F["n3"], DELTA / 3)))
        M = {k: np.mean(v) for k, v in R.items()}
        rows2.append(f"{name} & {M['rf']:.3f} & {M['mv']:.3f} & {M['gibbs']:.3f} & {M['dis']:.3f} & {int(M['n1'])} & {int(M['n2'])} & "
                     f"{M['fo']:.3f} & {M['tnd']:.3f} & {M['cb']:.3f} \\\\")
        rows3.append(f"{name} & {ms(R['mv'])} & {ms(R['best_u'])} & {ms(R['mv_fo'])} & {ms(R['cert_fo'])} & {np.mean(R['eff_fo']):.1f} & "
                     f"{ms(R['mv_tnd'])} & {ms(R['cert_tnd'])} & {np.mean(R['eff_tnd']):.1f} \\\\")
        summary["u"].append((M["mv"], M["best_u"])); summary["fo"].append((M["mv_fo"], M["cert_fo"])); summary["tnd"].append((M["mv_tnd"], M["cert_tnd"]))
        print(f"  {name}: mv {M['mv']:.3f} FO {M['fo']:.3f} TND {M['tnd']:.3f} CB {M['cb']:.3f} | FO-opt {M['mv_fo']:.3f}/{M['cert_fo']:.3f} TND-opt {M['mv_tnd']:.3f}/{M['cert_tnd']:.3f}")
    s = {k: np.mean(v, axis=0) for k, v in summary.items()}
    rows3.append(r"\midrule")
    rows3.append(f"mean & {s['u'][0]:.3f} & {s['u'][1]:.3f} & {s['fo'][0]:.3f} & {s['fo'][1]:.3f} & & {s['tnd'][0]:.3f} & {s['tnd'][1]:.3f} & \\\\")
    save_tab("p2_forest.tex", rows2); save_tab("p3_weights.tex", rows3)

## Part 2.1 -- Learning the posterior by minimizing a certificate

In [3]:
t0 = time.time(); part2_3(); print(f'{time.time() - t0:.0f} s')

Part 2 and 3


  australian: mv 0.141 FO 0.649 TND 0.819 CB 0.839 | FO-opt 0.170/0.509 TND-opt 0.138/0.772


  balance: mv 0.097 FO 0.523 TND 0.664 CB 0.731 | FO-opt 0.137/0.442 TND-opt 0.101/0.633


  bupa: mv 0.254 FO 1.000 TND 1.000 CB 1.000 | FO-opt 0.367/0.985 TND-opt 0.283/1.000


  german: mv 0.242 FO 0.864 TND 1.000 CB 0.968 | FO-opt 0.297/0.761 TND-opt 0.248/0.998


...

  abalone8: mv 0.141 FO 0.502 TND 0.591 CB 0.639 | FO-opt 0.178/0.472 TND-opt 0.143/0.572


  pageblocks: mv 0.021 FO 0.124 TND 0.150 CB 0.190 | FO-opt 0.034/0.105 TND-opt 0.022/0.140


  satimage: mv 0.062 FO 0.248 TND 0.271 CB 0.329 | FO-opt 0.094/0.227 TND-opt 0.061/0.258


  segmentation: mv 0.027 FO 0.210 TND 0.221 CB 0.310 | FO-opt 0.049/0.152 TND-opt 0.025/0.199


  yeast3: mv 0.046 FO 0.274 TND 0.352 CB 0.424 | FO-opt 0.053/0.220 TND-opt 0.048/0.325
  -> results/p2_forest.tex
  -> results/p3_weights.tex
144 s


## Part 2.2 -- Direct minimization of the kl certificate by gradient descent

In [4]:
from pbtools import certificate_and_grad


def adam_certificate(pi, n, kind, L=None, Lt=None, iters=300, lr=0.05):
    """Minimize the kl-form certificate over softmax logits with Adam."""
    th = np.log(pi).copy(); mo = np.zeros_like(th); ve = np.zeros_like(th); trace = []
    for it in range(iters):
        val, g = certificate_and_grad(th, pi, n, DELTA, kind, L=L, Lt=Lt)
        trace.append(val)
        mo = 0.9 * mo + 0.1 * g; ve = 0.999 * ve + 0.001 * g ** 2
        th -= lr * (mo / (1 - 0.9 ** (it + 1))) / (np.sqrt(ve / (1 - 0.999 ** (it + 1))) + 1e-8)
    return softmax(th), trace


def partB():
    print("Part B (gradient on the certificate)")
    rows = []; traces = {}
    for name in ["spambase", "satimage", "wdbc", "pageblocks"]:
        X, y = load(name); R = []
        for seed in range(3):
            F = forest_stats(X, y, seed)
            r_alt = optimize_tnd(F["pi"], F["Lt"], F["n2"], DELTA)
            r_gd, tr = adam_certificate(F["pi"], F["n2"], "tnd", Lt=F["Lt"])
            if seed == 0:
                traces[name] = tr
            R.append((tnd_bound(F["pi"], F["pi"], F["Lt"], F["n2"], DELTA), mv_statistics(F["Vte"], F["yte"], F["pi"])["mv"],
                      tnd_bound(r_alt, F["pi"], F["Lt"], F["n2"], DELTA), mv_statistics(F["Vte"], F["yte"], r_alt)["mv"], eff(r_alt),
                      tnd_bound(r_gd, F["pi"], F["Lt"], F["n2"], DELTA), mv_statistics(F["Vte"], F["yte"], r_gd)["mv"], eff(r_gd)))
        R = np.mean(R, 0)
        rows.append(f"{name} & {R[0]:.3f} & {R[1]:.3f} & {R[2]:.3f} & {R[3]:.3f} & {R[4]:.0f} & {R[5]:.3f} & {R[6]:.3f} & {R[7]:.0f} \\\\")
    save_tab("pB_gradient.tex", rows)
    fig, ax = plt.subplots(figsize=(5.5, 2.8))
    for name, tr in traces.items():
        ax.plot(tr, label=name)
    ax.set_xlabel("Adam iterations"); ax.set_ylabel("TND certificate"); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(FIG, "tp_gradient.pdf")); plt.close(fig)

In [5]:
t0 = time.time(); partB(); print(f'{time.time() - t0:.0f} s')

Part B (gradient on the certificate)


  -> results/pB_gradient.tex


19 s


## Part 2.3 -- Strength versus diversity

In [6]:
def part5():
    print("Part 5")
    rows = []
    for name in ["spambase", "german"]:
        X, y = load(name); d = X.shape[1]
        for mf in [1, "sqrt", 0.5, None]:
            R = []
            for seed in range(3):
                F = forest_stats(X, y, seed, max_features=mf)
                st = mv_statistics(F["Vte"], F["yte"], F["pi"])
                R.append((st["gibbs"], st["dis"], st["mv"], tnd_bound(F["pi"], F["pi"], F["Lt"], F["n2"], DELTA),
                          cbound_pb(F["pi"], F["pi"], F["L"], F["n1"], F["D"], F["n3"], DELTA)))
            R = np.mean(R, 0)
            lab = {1: "1", "sqrt": r"$\sqrt d$", 0.5: r"$d/2$", None: r"$d$"}[mf]
            rows.append(f"{name} & {lab} & {R[0]:.3f} & {R[1]:.3f} & {R[2]:.3f} & {R[3]:.3f} & {R[4]:.3f} \\\\")
    save_tab("p5_diversity.tex", rows)

In [7]:
t0 = time.time(); part5(); print(f'{time.time() - t0:.0f} s')

Part 5


  -> results/p5_diversity.tex
24 s


## Part 2.4 -- Self-bounding model selection versus cross-validation

In [8]:
def part6():
    print("Part 6")
    depths = [1, 2, 4, 8, None]; K = len(depths); rows = []
    for name in ["spambase", "pima", "wdbc", "german"]:
        X, y = load(name); sel_c, sel_cv, te_c, te_cv, te_best, cert_sel = [], [], [], [], [], []
        for seed in range(3):
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
            certs, tests = [], []
            for dp in depths:
                bt = BaggedTrees(100, bootstrap_frac=0.5, max_depth=dp, random_state=seed).fit(Xtr, ytr)
                Lt, n2 = bt.oob_tandem(Xtr, ytr); pi = np.ones(100) / 100
                rho = optimize_tnd(pi, Lt, n2, DELTA / K)
                certs.append(tnd_bound(rho, pi, Lt, n2, DELTA / K)); tests.append(np.mean(bt.predict(Xte, rho) != yte))
            # 5-fold cross-validation of the uniform forest
            cv = []
            for dp in depths:
                errs = []
                for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(Xtr, ytr):
                    bt = BaggedTrees(100, bootstrap_frac=0.5, max_depth=dp, random_state=seed).fit(Xtr[a], ytr[a])
                    errs.append(np.mean(bt.predict(Xtr[b]) != ytr[b]))
                cv.append(np.mean(errs))
            ic, iv = int(np.argmin(certs)), int(np.argmin(cv))
            sel_c.append(ic); sel_cv.append(iv); te_c.append(tests[ic]); te_cv.append(tests[iv]); te_best.append(min(tests))
            cert_sel.append(certs[ic])
        lab = lambda i: str(depths[i]) if depths[i] else "None"
        rows.append(f"{name} & {', '.join(lab(i) for i in sel_c)} & {np.mean(te_c):.3f} & {', '.join(lab(i) for i in sel_cv)} & {np.mean(te_cv):.3f} & {np.mean(te_best):.3f} & {np.mean(cert_sel):.3f} \\\\")
    save_tab("p6_selection.tex", rows)

In [9]:
t0 = time.time(); part6(); print(f'{time.time() - t0:.0f} s')

Part 6


  -> results/p6_selection.tex
46 s
